# 26 · Load: the industry → job bridge (OEWS research estimates for New York, BLS national matrix) → `staging`

**Source cards:** `bls_oews_research.md`, `bls_ind_occ_matrix.md` (§6 issues, §7 staging tables). Explored in: notebook 09. Decision: D3 (the NY research estimates are the bridge; the national matrix is for the 2035 view and self-employment).

| Table | One row = | Expected rows |
|---|---|---|
| `stg_bls__oews_research_ny` | year × industry (any NAICS level) × job, New York State | 204,386 |
| `stg_bls__ind_occ_matrix` | industry × job, United States, 2025 and 2035 | 113,898 |

Both are research estimates / national and are labeled that way wherever they're used. The seven research workbooks hold every state (~850k rows each), so they're streamed and only New York is kept: about 75 s per file.

In [1]:
import sys
import time
from pathlib import Path

import openpyxl
import pandas as pd
from IPython.display import display

PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "common.py").exists())
sys.path.insert(0, str(PROJECT / "src"))
from common import log
from staging import connect, raw_files, stage

pd.set_option("display.max_columns", 40, "display.max_colwidth", 70, "display.width", 220)
NB = "26-load-industry-job-bridge.ipynb"
conn = connect()
summary = []

## 1. OEWS research estimates: stream each year, keep New York (issue 1)

Header names, column order and the data sheet's name change by year (2019: `occ code`, `i_group` after `o_group`; 2024 adds `naics2` and `release`), so headers are lower-cased with `_` and columns are picked by name. The data sheet is always the first one.

In [2]:
def read_ny(path: Path) -> tuple[pd.DataFrame, int]:
    t = time.time()
    wb = openpyxl.load_workbook(path, read_only=True)
    it = wb[wb.sheetnames[0]].iter_rows(values_only=True)
    head = [str(h).strip().lower().replace(" ", "_") for h in next(it)]
    ai = head.index("area")
    rows, n = [], 0
    for r in it:
        n += 1
        if str(r[ai]).strip() == "36":
            rows.append(["" if v is None else str(v).strip() for v in r])
    wb.close()
    print(f"  {path.name}: {n:,} rows -> {len(rows):,} New York rows ({time.time() - t:.0f}s)")
    return pd.DataFrame(rows, columns=head), n

COLS = ["naics", "naics_title", "i_group", "occ_code", "occ_title", "o_group", "tot_emp", "emp_prse", "pct_total",
        "h_median", "a_median", "a_mean"]
rf = raw_files("bls_oews_research")
parts, n_in = [], 0
for f, p in zip(rf.file, rf.path):
    year = int(Path(f).stem.split("_")[2])
    d, n = read_ny(p)
    n_in += n
    missing = [c for c in COLS if c not in d.columns]
    assert not missing, f"{year}: missing {missing}"
    parts.append(d[COLS].assign(year=year, source_file=f))
r = pd.concat(parts, ignore_index=True)
log("all states -> New York (area 36), 2019-2025", n_in, len(r))

  oes_research_2019_allsectors.xlsx: 836,739 rows -> 29,349 New York rows (68s)


  oes_research_2020_allsectors.xlsx: 816,060 rows -> 28,274 New York rows (71s)


  oes_research_2021_allsectors.xlsx: 818,972 rows -> 28,335 New York rows (74s)


  oes_research_2022_allsectors.xlsx: 835,717 rows -> 28,792 New York rows (76s)


  oes_research_2023_allsectors.xlsx: 855,397 rows -> 29,505 New York rows (77s)


  oes_research_2024_allsectors.xlsx: 861,922 rows -> 29,887 New York rows (85s)


  oes_research_2025_allsectors.xlsx: 867,905 rows -> 30,244 New York rows (77s)
  all states -> New York (area 36), 2019-2025: 5,892,712 -> 204,386 rows (-5,688,326)


## 2. Symbols → NULL + flags (issue 4); vintages (issue 3)

`**` = employment not published, `~` = share under 0.05%, `#` = wage top-coded, `*` = wage not published. Each becomes NULL with a flag (`emp_suppressed`, `pct_lt_0_05`, `wage_topcoded`). Codes: hybrid SOC in 2019–20, SOC 2018 from 2021; NAICS 2017 to 2021, NAICS 2022 from 2022. The same NAICS code can sit at two levels, so the key includes `i_group` (issue 2).

In [3]:
SYM = {"**", "*", "#", "~", ""}
r["emp_suppressed"] = r.tot_emp.eq("**")
r["pct_lt_0_05"] = r.pct_total.eq("~")
r["wage_topcoded"] = r[["h_median", "a_median", "a_mean"]].eq("#").any(axis=1)
counts = {}
for c in ["tot_emp", "emp_prse", "pct_total", "h_median", "a_median", "a_mean"]:
    v = r[c].str.replace(",", "", regex=False)
    sym = v.isin(SYM)
    bad = pd.to_numeric(v[~sym], errors="coerce").isna()
    assert not bad.any(), f"{c}: unexpected text {v[~sym][bad].unique()[:5]}"
    counts[c] = v[sym].replace("", "(blank)").value_counts().to_dict()
    r[c] = pd.to_numeric(v.mask(sym), errors="raise").astype("float64")
print("Symbols -> NULL, by column:", counts)
assert (r.tot_emp.dropna() % 1 == 0).all()
r["tot_emp"] = r.tot_emp.astype("Int64")
r["year"] = r.year.astype("Int64")
r["soc_vintage"] = r.year.map(lambda y: "hybrid" if y <= 2020 else "2018")
r["naics_vintage"] = r.year.map(lambda y: "2017" if y <= 2021 else "2022")
print(r.groupby("year").size().to_dict())
print("Industry levels:", r.i_group.value_counts().to_dict(), "| job levels:", r.o_group.value_counts().to_dict())

stg_r = r[["year", "naics", "naics_title", "i_group", "occ_code", "occ_title", "o_group", "tot_emp", "emp_prse", "pct_total",
           "h_median", "a_median", "a_mean", "emp_suppressed", "pct_lt_0_05", "wage_topcoded", "soc_vintage", "naics_vintage",
           "source_file"]].sort_values(["year", "naics", "i_group", "occ_code"]).reset_index(drop=True)
summary.append(stage(stg_r, "stg_bls__oews_research_ny", ["year", "naics", "i_group", "occ_code"], expected_rows=204_386,
                     source_key="bls_oews_research", notebook=NB, conn=conn,
                     comment="BLS OEWS research estimates, New York State, industry x job, May 2019-2025. Research estimates, not official. D3 bridge."))

Symbols -> NULL, by column: {'tot_emp': {'**': 21379}, 'emp_prse': {'**': 21379}, 'pct_total': {'**': 21503, '~': 6234}, 'h_median': {'*': 4178, '#': 1831}, 'a_median': {'#': 1838, '*': 1771}, 'a_mean': {'*': 1771, '#': 196}}
{np.int64(2019): 29349, np.int64(2020): 28274, np.int64(2021): 28335, np.int64(2022): 28792, np.int64(2023): 29505, np.int64(2024): 29887, np.int64(2025): 30244}
Industry levels: {'4-digit': 97088, '3-digit': 57259, 'sector': 30771, '5-digit': 17985, '6-digit': 1283} | job levels: {'detailed': 169584, 'major': 31978, 'total': 2824}


  stg_bls__oews_research_ny: 204,386 rows x 19 cols, key ['year', 'naics', 'i_group', 'occ_code'] unique | local data/interim/staging/stg_bls__oews_research_ny.parquet | DB: 204386


## 3. BLS National Employment Matrix, 2025–35 (issues 1–6 of its card)

Every numeric cell is a plain number (nb 09). Employment in thousands. `naics_sector` is filled for the sector rows only (`NN0000`, `NN-NN0`, and `491000` Postal Service, which the matrix keeps outside 48-49); line-item industries don't add up to their sectors, so the bridge uses sector rows.

In [4]:
mf = raw_files("bls_ind_occ_matrix").iloc[-1]
m = pd.read_excel(mf.path, sheet_name="Matrix", dtype=str, keep_default_na=False).apply(lambda s: s.str.strip())
log("Matrix sheet", len(m), len(m))
REN = {"Industry code": "industry_code", "Industry title": "industry_title", "Industry type": "industry_type",
       "Occupation code": "occ_code", "Occupation title": "occ_title", "Occupation type": "occ_type",
       "2025 Employment": "emp_2025_k", "2025 Percent of Industry": "pct_of_industry_2025",
       "2025 Percent of Occupation": "pct_of_occ_2025", "2035 Employment": "emp_2035_k",
       "2035 Percent of Industry": "pct_of_industry_2035", "2035 Percent of Occupation": "pct_of_occ_2035"}
assert set(REN) <= set(m.columns), set(REN) - set(m.columns)
m = m[list(REN)].rename(columns=REN)
for c in [c for c in m.columns if c[:3] in ("emp", "pct")]:
    m[c] = pd.to_numeric(m[c], errors="raise").astype("float64")

def sector(code: str):
    if code == "491000":
        return "491"                          # Postal Service, its own row in the matrix
    if len(code) == 6 and code.isdigit() and code.endswith("0000"):
        return code[:2]
    if len(code) == 6 and code[2] == "-" and code.endswith("0"):
        return f"{code[:2]}-{code[3:5]}"      # 31-330 -> 31-33
    return None

m["naics_sector"] = m.industry_code.map(sector)
sec = m.drop_duplicates("industry_code").dropna(subset=["naics_sector"])[["industry_code", "naics_sector", "industry_title"]]
display(sec.sort_values("naics_sector").reset_index(drop=True))
tot = m[(m.industry_code == "TE1000") & (m.occ_code == "00-0000")].emp_2025_k.iloc[0]
print(f"All industries, all jobs, 2025: {tot:,.1f}k (Table 1.2: 170,280.8k)"); assert tot == 170280.8
stg_m = m.assign(edition="2025-2035", source_file=mf.file).sort_values(["industry_code", "occ_code"]).reset_index(drop=True)
summary.append(stage(stg_m, "stg_bls__ind_occ_matrix", ["industry_code", "occ_code"], expected_rows=113_898,
                     source_key="bls_ind_occ_matrix", notebook=NB, conn=conn,
                     comment="BLS National Employment Matrix 2025-35, industry x job (thousands). United States. naics_sector on sector rows only."))

  Matrix sheet: 113,898 -> 113,898 rows (+0)


,industry_code,naics_sector,industry_title
0,110000,11,"Agriculture, forestry, fishing, and hunting"
1,210000,21,"Mining, quarrying, and oil and gas extraction"
2,220000,22,Utilities
3,230000,23,Construction
4,31-330,31-33,Manufacturing
5,420000,42,Wholesale trade
6,44-450,44-45,Retail trade
7,48-490,48-49,Transportation and warehousing
8,491000,491,Postal service
9,510000,51,Information


All industries, all jobs, 2025: 170,280.8k (Table 1.2: 170,280.8k)


  stg_bls__ind_occ_matrix: 113,898 rows x 15 cols, key ['industry_code', 'occ_code'] unique | local data/interim/staging/stg_bls__ind_occ_matrix.parquet | DB: 113898


## 4. Summary

In [5]:
display(pd.concat(summary, ignore_index=True))
if conn is not None:
    conn.close()

,table,rows,cols,key,source_key,notebook,in_db,loaded_utc
0,stg_bls__oews_research_ny,204386,19,year + naics + i_group + occ_code,bls_oews_research,26-load-industry-job-bridge.ipynb,204386,2026-10-14T21:21:49+00:00
1,stg_bls__ind_occ_matrix,113898,15,industry_code + occ_code,bls_ind_occ_matrix,26-load-industry-job-bridge.ipynb,113898,2026-10-14T21:21:57+00:00
